Preparacao de dados
- Textos precisam ser normalizados (patronizados)
Em geral o pre-processamento de textos envolve passos como:
1. Tokenizacao: Quebra do texto em unidades menores (Tokens) -> ["O", "servidor", "Foi"] ou ["Ama", "nha", "eu", "pre", "tendo"] {Existem muitas abordagens}
2. Lowercasing: SERVIDOR -> servidor (Esse tipo pode nao ser interessante para algumas tarefas onde a caixa alta pode representar sentimentos em determinados contextos)
3. Remocao de stopwords: Palavras frequentes que nao ajudam muito, conectivos que nao expressao muito significado semantico. (o, foi, por, que, a ,etc.)
4. Stemming: Reduz a palavra para a sua RAIZ, ou seja, remove a conjugacao. (promovido -> promover, promocao -> promover)
5. Lemmatizacao: Tenta reduzir a palavra para uma forma canônica linguística mais válida
5. Remover pontuacao
6. Remover números
7. Normalizar Acentos
8. Remocao de Tag HTML
9. Remocao de URLs
10. Correcao Ortografica
11. Remocao de Emojies

Baixando o Dataset pra rodar as paradas

In [2]:
import kagglehub
from jupyter_core.version import pattern

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1


Cola o dataset aqui na pasta do projeto e poe o path ai na variavel

In [8]:
PATH = r"C:\Users\User\Documents\datasets\tabulate\IMDB Dataset.csv" #r é pra nao interpreter escapes, assim vira texto literal

In [16]:
import pandas as pd

df = pd.read_csv(PATH)

print(df.columns)
print(df.count())

Index(['review', 'sentiment'], dtype='str')
review       50000
sentiment    50000
dtype: int64


In [17]:
df = df[['review', 'sentiment']].copy()
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [22]:
# so ficar olhando durante o processo
sample_text = df.loc[0, 'review']
print(sample_text[:300])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


Convertertendo pra lowercase (minisculo)

In [26]:
df['review_lower'] = df['review'].str.lower()
print(df['review_lower'].iloc[0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. tru


Removendo Tags HTML

In [27]:
import re

def remove_html_tags(text):
    regex = re.compile(r'<.*?>')
    return re.sub(regex, '', text)

df['review_no_html'] = df['review_lower'].apply(remove_html_tags)
print(df['review_no_html'].iloc[0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this 


Removendo URL e links

In [32]:
def remove_urls(text):
    regex = re.compile(r'https?://\S+|www\.\S+')
    return regex.sub('', text)

df['review_no_url'] = df['review_no_html'].apply(remove_urls)
print(df['review_no_url'].iloc[0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this 


Remocao de pontuacao

In [34]:
import string

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['review_no_punct'] = df['review_no_url'].apply(remove_punctuation)
print(df['review_no_punct'].iloc[0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not 


Chat Words (texto de chat cheio de abreviacoes da geracao z)

In [35]:
chat_words = {
    "u": "you",
    "ur": "your",
    "lol": "laughing out loud",
    "idk": "i do not know",
    "omg": "oh my god",
    "btw": "by the way",
    "imo": "in my opinion",
    "imho": "in my humble opinion"
}

def replace_chat_words(text):
    words = text.split()
    expanded = [chat_words.get(word, word) for word in words]
    return " ".join(expanded)

df['review_chat_fixed'] = df['review_no_punct'].apply(replace_chat_words)
print(df['review_chat_fixed'].iloc[0][:300])

one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not 


Correacao de ortografia

In [40]:
from textblob import TextBlob

def correct_spelling(text):
    return str(TextBlob(text).correct())

# é lento pakas e muitas vezes meio desnecessario, ou ate piora o resultado, entao nao usam muito.
df_sample = df.head(5).copy()
df_sample['review_spelling_corrected'] = df_sample['review_chat_fixed'].apply(correct_spelling)

print(df_sample[['review_chat_fixed', 'review_spelling_corrected']].head(3))

                                   review_chat_fixed  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production the filming tech...   
2  i thought this was a wonderful way to spend ti...   

                           review_spelling_corrected  
0  one of the other reviews has mentioned that af...  
1  a wonderful little production the filling tech...  
2  i thought this was a wonderful way to spend ti...  


Remocao de stop words (the, is, and, of, etc)

In [43]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in stop_words]
    return " ".join(filtered)

df['review_no_stopwords'] = df['review_chat_fixed'].apply(remove_stopwords)
print(df['review_no_stopwords'].iloc[0][:300])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


one reviewers mentioned watching 1 oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordit called oz nickname given oswa


handling emojis

In [45]:
import emoji

def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

df['review_no_emoji'] = df['review_no_stopwords'].apply(remove_emojis)
print(df['review_no_emoji'].iloc[0][:300])

one reviewers mentioned watching 1 oz episode youll hooked right exactly happened methe first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordit called oz nickname given oswa


Tokenizacao

In [48]:
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

df['tokens'] = df['review_no_emoji'].apply(word_tokenize)
print(df['tokens'].iloc[0][:300])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'show', 'faint', 'hearted', 'timid', 'show', 'pulls', 'punches', 'regards', 'drugs', 'sex', 'violence', 'hardcore', 'classic', 'use', 'wordit', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focuses', 'mainly', 'emerald', 'city', 'experimental', 'section', 'prison', 'cells', 'glass', 'fronts', 'face', 'inwards', 'privacy', 'high', 'agenda', 'em', 'city', 'home', 'manyaryans', 'muslims', 'gangstas', 'latinos', 'christians', 'italians', 'irish', 'moreso', 'scuffles', 'death', 'stares', 'dodgy', 'dealings', 'shady', 'agreements', 'never', 'far', 'awayi', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'goes', 'shows', 'wouldnt', 'dare', 'forget', 'pretty', 'pictures', 'painted', 'mainst

stemming (reduzir palavras a um radical aproximado como: liked, likes, liking para algo como like ou uma forma truncada, dependendo do stemmer)

In [49]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stem_words(tokens):
    return [ps.stem(word) for word in tokens]

df['tokens_stemmed'] = df['tokens'].apply(stem_words)
print(df['tokens_stemmed'].iloc[0][:100])

['one', 'review', 'mention', 'watch', '1', 'oz', 'episod', 'youll', 'hook', 'right', 'exactli', 'happen', 'meth', 'first', 'thing', 'struck', 'oz', 'brutal', 'unflinch', 'scene', 'violenc', 'set', 'right', 'word', 'go', 'trust', 'show', 'faint', 'heart', 'timid', 'show', 'pull', 'punch', 'regard', 'drug', 'sex', 'violenc', 'hardcor', 'classic', 'use', 'wordit', 'call', 'oz', 'nicknam', 'given', 'oswald', 'maximum', 'secur', 'state', 'penitentari', 'focus', 'mainli', 'emerald', 'citi', 'experiment', 'section', 'prison', 'cell', 'glass', 'front', 'face', 'inward', 'privaci', 'high', 'agenda', 'em', 'citi', 'home', 'manyaryan', 'muslim', 'gangsta', 'latino', 'christian', 'italian', 'irish', 'moreso', 'scuffl', 'death', 'stare', 'dodgi', 'deal', 'shadi', 'agreement', 'never', 'far', 'awayi', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'goe', 'show', 'wouldnt', 'dare', 'forget', 'pretti', 'pictur']


lematização (lematization) - tenta reduzir a palavra para uma forma canônica linguística mais válida.

In [50]:
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df['tokens_lemmatized'] = df['tokens'].apply(lemmatize_words)
print(df['tokens_lemmatized'].iloc[0])

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...


['one', 'reviewer', 'mentioned', 'watching', '1', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'methe', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'show', 'faint', 'hearted', 'timid', 'show', 'pull', 'punch', 'regard', 'drug', 'sex', 'violence', 'hardcore', 'classic', 'use', 'wordit', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focus', 'mainly', 'emerald', 'city', 'experimental', 'section', 'prison', 'cell', 'glass', 'front', 'face', 'inwards', 'privacy', 'high', 'agenda', 'em', 'city', 'home', 'manyaryans', 'muslim', 'gangsta', 'latino', 'christian', 'italian', 'irish', 'moreso', 'scuffle', 'death', 'stare', 'dodgy', 'dealing', 'shady', 'agreement', 'never', 'far', 'awayi', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'go', 'show', 'wouldnt', 'dare', 'forget', 'pretty', 'picture', 'painted', 'mainstream', 'audience', 'forg

comparando original com coluna final consolidada, com tokens lematizados de volta em texto.

In [51]:
df['clean_review'] = df['tokens_lemmatized'].apply(lambda x: " ".join(x))
print(df[['review', 'clean_review']].head(3))

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   

                                        clean_review  
0  one reviewer mentioned watching 1 oz episode y...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
